# Filter Atoms

Clean up extracted atoms from chesspublishing using a second LLM pass.

Actions (in priority order):
1. **Keep** — Concrete, verifiable positional facts about the move
2. **Contextualize** — Not self-contained; rewrite with missing context so it stands alone (keep as separate atom)
3. **Move to alternative** — About a different move, not the played one
4. **Remove** — Generic labels only (no positional content)

In [ ]:
import json
import os
import random
import chess
import chess.svg
import openai
from IPython.display import display, SVG, Markdown, clear_output
from dotenv import load_dotenv

load_dotenv()

DIR = os.path.dirname(os.path.abspath('__file__'))
INPUT_PATH = os.path.join(DIR, 'data', 'extraction_included.jsonl')

with open(INPUT_PATH) as f:
    all_rows = [json.loads(line) for line in f]

print(f'Loaded {len(all_rows)} positions from {INPUT_PATH}')

total_atoms = sum(len(r['extracted'].get('reasoning', [])) for r in all_rows)
print(f'Total atoms: {total_atoms}')

client = openai.OpenAI()

In [ ]:
def parse_json(text):
    """Parse JSON from LLM output, stripping markdown code fences if present."""
    text = text.strip()
    if text.startswith('```'):
        text = text.split('\n', 1)[1]
        if text.endswith('```'):
            text = text[:-3]
        text = text.strip()
    return json.loads(text)

---
# Part 1 — Filter pass

Classify each reasoning atom as keep / contextualize / move_to_alternative / remove.

Full position context (FEN, commentary, variation) is
sent to the LLM. Only reasoning atoms are classified.

In [ ]:
FILTER_MODEL = 'gpt-5.4'

FILTER_PROMPT = """\
You are filtering extracted reasoning atoms from chess commentary.

You will receive the FULL extracted JSON for a position (FEN, move played,
original commentary, reasoning atoms, etc.)

Classify each REASONING atom as: keep / contextualize / move_to_alternative / remove.

KEEP an atom if it states a concrete, verifiable positional fact about what
the move does: attacks, defends, controls, pins, blocks, develops, opens/
closes lines, creates threats, prevents opponent plans, explains why a move
is bad, etc.

CONTEXTUALIZE an atom that has useful content but is not self-contained on
its own. An atom is not self-contained if it references a consequence,
position, or piece without enough context to verify it independently.

Each atom stays SEPARATE — do NOT combine multiple atoms into one. Instead,
add the missing context so each atom is independently verifiable.
  Original: "On e4 the knight controls d6 and f6"
  Contextualized: "After Nd2, the knight on e4 would control d6 and f6."
Output: {"action": "contextualize", "new_text": "After Nd2, ..."}

IMPORTANT: Only add REFERENTIAL context — which move, which piece, which
square, which position. Do NOT add new chess claims, analysis, or
consequences that were not in the original atom. The goal is to make the
existing claim self-contained, not to enrich it.

MOVE_TO_ALTERNATIVE if the atom describes an ALTERNATIVE move (a move NOT
played) and its consequences.
  Output: {"action": "move_to_alternative", "move": "Bf4", "text": "..."}
  You may paraphrase the text so it reads naturally.

Conclusion vs. detailed analysis of alternatives:
- A CONCLUSION that the played move avoids or is better than an alternative
  is KEEP: "Qh4 avoids the inferior Qg5" -> KEEP.
- A DETAILED ANALYSIS of what happens after the alternative move is
  MOVE_TO_ALTERNATIVE: "After 32.Qc1 Nxd4 33.Bxd4 Rxd4, Black threatens
  Rd3" -> MOVE_TO_ALTERNATIVE for Qc1.
- When an atom MIXES both (conclusion + detailed line), SPLIT it using
  kept_brief:
  {"action": "move_to_alternative", "move": "Qc1",
   "text": "After 32.Qc1 Nxd4 33.Bxd4 Rxd4, Black threatens Rd3.",
   "kept_brief": "Qb2 avoids Qc1, which leads to dangerous threats on the d-file."}
  kept_brief stays in reasoning, text goes to the alternative.

REMOVE an atom ONLY if it is a generic label with no concrete positional
content. "Nbd2 is a typical Colle Attack manoeuvre." — naming an opening or
saying "typical/standard" without any positional claim is not a useful atom.

─── OUTPUT FORMAT ───

Output ONLY valid JSON:
{"atoms": [
  {"index": 0, "text": "...", "action": "keep"},
  {"index": 1, "text": "...", "action": "contextualize", "new_text": "..."},
  {"index": 2, "text": "...", "action": "move_to_alternative", "move": "Qc1",
   "text": "detailed line...", "kept_brief": "brief conclusion..."},
  {"index": 3, "text": "...", "action": "remove", "reason": "generic label"}
 ]
}

Be conservative: when in doubt, KEEP. Prefer CONTEXTUALIZE over REMOVE when
an atom has useful content but just lacks context. Prefer MOVE_TO_ALTERNATIVE
over REMOVE when the atom is about a different move.
"""

print(f'Filter model: {FILTER_MODEL}')
print(f'Filter prompt: {len(FILTER_PROMPT)} chars')

In [ ]:
def filter_atoms(row, model=FILTER_MODEL):
    """Classify each reasoning atom as keep/remove/contextualize/move_to_alternative."""
    extracted = row['extracted']
    reasoning = extracted.get('reasoning', [])

    all_atoms = [{'index': i, 'text': a} for i, a in enumerate(reasoning)]

    if not all_atoms:
        return {'atoms': [], 'filtered_reasoning': [],
                'alternatives': []}

    # Full context for the LLM
    context = {
        'fen': row['fen'],
        'move_uci': row['move_uci'],
        'move_san': row.get('move_san', ''),
        'annotation': row['annotation'],
        'is_mainline': row.get('is_mainline', True),
        'parent_comment': row.get('parent_comment', ''),
        'reasoning': reasoning,
    }
    # Include variation if present
    if extracted.get('variation'):
        context['variation'] = extracted['variation']

    atoms_text = '\n'.join(f'{a["index"]}. \"{a["text"]}\"' for a in all_atoms)
    user_msg = (
        f'Position data:\n```json\n{json.dumps(context, indent=2)}\n```\n\n'
        f'Reasoning atoms to classify:\n{atoms_text}'
    )

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': FILTER_PROMPT},
            {'role': 'user', 'content': user_msg},
        ],
        temperature=0,
        max_completion_tokens=2048,
    )
    text = resp.choices[0].message.content.strip()

    try:
        result = parse_json(text)
    except json.JSONDecodeError:
        return {'atoms': [], 'error': 'JSON parse error', 'raw': text,
                'filtered_reasoning': reasoning, 'alternatives': []}

    if 'atoms' not in result:
        return {'atoms': [], 'error': 'no atoms key',
                'filtered_reasoning': reasoning, 'alternatives': []}

    # Parse reasoning atom actions
    atoms_by_idx = {a['index']: a for a in result['atoms']}
    alternatives = []

    # Build filtered reasoning
    filtered_reasoning = []
    for atom_info in all_atoms:
        idx = atom_info['index']
        a = atoms_by_idx.get(idx, {})
        action = a.get('action', 'keep')

        if action == 'remove':
            continue
        if action == 'move_to_alternative':
            alternatives.append({
                'move': a.get('move', '?'),
                'text': a.get('text', ''),
            })
            # If kept_brief provided, add that to reasoning instead
            if a.get('kept_brief'):
                filtered_reasoning.append(a['kept_brief'])
            continue
        if action == 'contextualize' and a.get('new_text'):
            filtered_reasoning.append(a['new_text'])
        else:
            filtered_reasoning.append(atom_info['text'])

    return {
        'atoms': result['atoms'],
        'filtered_reasoning': filtered_reasoning,
        'alternatives': alternatives,
    }


def run_filter(row, model=FILTER_MODEL):
    """Run filter and build the output row."""
    result = filter_atoms(row, model=model)

    out_row = dict(row)
    filtered_extracted = dict(row['extracted'])
    filtered_extracted['reasoning'] = result['filtered_reasoning']

    # Store alternatives if any were produced
    alts = result['alternatives']
    if alts:
        # Group by move
        alt_by_move = {}
        for a in alts:
            move = a['move']
            if move not in alt_by_move:
                alt_by_move[move] = {'move': move, 'reasoning': []}
            alt_by_move[move]['reasoning'].append(a['text'])
        alt_list = list(alt_by_move.values())
        filtered_extracted['alternative'] = alt_list[0] if len(alt_list) == 1 else alt_list

    out_row['extracted'] = filtered_extracted

    return result, out_row


print('filter_atoms, run_filter ready')

### Filter sample: 20 positions

In [ ]:
SEED = 45
N_SAMPLE = 20

rng = random.Random(SEED)
review_order = list(range(len(all_rows)))
rng.shuffle(review_order)

sample_rows = [all_rows[i] for i in review_order[:N_SAMPLE]]

sample_results = []
for i, row in enumerate(sample_rows):
    board = chess.Board(row['fen'])
    san = board.san(chess.Move.from_uci(row['move_uci']))
    print(f'[{i+1}/{N_SAMPLE}] {row["game"]} \u2014 {san}')
    result, out_row = run_filter(row)
    sample_results.append((row, result, out_row))

print(f'\nDone. {N_SAMPLE} positions filtered.')

In [ ]:
for idx, (row, result, out_row) in enumerate(sample_results):
    board = chess.Board(row['fen'])
    move = chess.Move.from_uci(row['move_uci'])
    move_san = board.san(move)

    svg = chess.svg.board(
        board,
        arrows=[(move.from_square, move.to_square)],
        size=300,
    )
    display(SVG(svg))

    turn = 'White' if board.turn == chess.WHITE else 'Black'
    loc = 'mainline' if row.get('is_mainline') else 'variation'
    display(Markdown(
        f'### [{idx+1}/{len(sample_results)}] {row["game"]} \u2014 '
        f'{turn} plays {move_san} ({loc})'
    ))

    display(Markdown(f'> {row["annotation"]}'))

    if row.get('parent_comment'):
        display(Markdown(f'*Context: {row["parent_comment"]}*'))

    # Show LLM decisions per reasoning atom
    atoms = result.get('atoms', [])
    if atoms:
        lines = []
        for a in atoms:
            action = a.get('action', 'keep')
            if action == 'keep':
                lines.append(f'\u2705 {a["text"]}')
            elif action == 'remove':
                reason = f' \u2014 *{a["reason"]}*' if a.get('reason') else ''
                lines.append(f'\u274c {a["text"]}{reason}')
            elif action == 'move_to_alternative':
                alt_move = a.get('move', '?')
                lines.append(f'\u27a1\ufe0f {a["text"]} \u2192 *alternative ({alt_move})*')
                if a.get('kept_brief'):
                    lines.append(f'  \u2192 kept brief: **{a["kept_brief"]}**')
            elif action == 'contextualize':
                lines.append(f'\U0001f504 {a["text"]}')
                if a.get('new_text'):
                    lines.append(f'  \u2192 **{a["new_text"]}**')
        display(Markdown('#### Atom decisions\n' + '\n\n'.join(lines)))

    # Show resulting reasoning
    new_r = result['filtered_reasoning']
    if new_r:
        display(Markdown('#### Resulting reasoning\n' +
                         '\n'.join(f'{i+1}. {a}' for i, a in enumerate(new_r))))

    # Show alternatives
    new_alt = out_row['extracted'].get('alternative')
    if new_alt:
        alts = new_alt if isinstance(new_alt, list) else [new_alt]
        alt_blocks = []
        for a in alts:
            block = f'**{a["move"]}**:\n'
            for r in a.get('reasoning', []):
                block += f'  - {r}\n'
            alt_blocks.append(block)
        display(Markdown('#### Alternatives\n\n' + '\n'.join(alt_blocks)))

    # Show variation if present
    var = row['extracted'].get('variation')
    if var:
        display(Markdown(f'#### Variation\n`{var}`'))

    old_r = row['extracted'].get('reasoning', [])
    n_moved = len(result.get('alternatives', []))
    n_ctx = sum(1 for a in atoms if a.get('action') == 'contextualize')
    n_removed = sum(1 for a in atoms if a.get('action') == 'remove')
    n_brief = sum(1 for a in atoms if a.get('kept_brief'))

    summary = f'**{len(old_r)} \u2192 {len(new_r)} reasoning atoms'
    parts = []
    if n_removed > 0:
        parts.append(f'{n_removed} removed')
    if n_ctx > 0:
        parts.append(f'{n_ctx} contextualized')
    if n_moved > 0:
        parts.append(f'{n_moved} \u2192 alternative')
    if n_brief > 0:
        parts.append(f'{n_brief} kept brief')
    if parts:
        summary += f' ({", ".join(parts)})'
    summary += '**'
    display(Markdown(summary))

    display(Markdown('---'))

In [ ]:
total_before = 0
total_after = 0
removed_reasons = []
ctx_count = 0
moved_count = 0

for row, result, out_row in sample_results:
    old = row['extracted'].get('reasoning', [])
    total_before += len(old)
    total_after += len(result['filtered_reasoning'])
    for a in result.get('atoms', []):
        action = a.get('action', 'keep')
        if action == 'remove':
            removed_reasons.append(a.get('reason', '?'))
        elif action == 'contextualize':
            ctx_count += 1
        elif action == 'move_to_alternative':
            moved_count += 1

net_change = total_before - total_after
print(f'Sample: {total_before} reasoning atoms -> {total_after} ({net_change} net removed, {net_change/max(total_before,1)*100:.1f}%)')
print(f'  Contextualized: {ctx_count}')
print(f'  Moved to alternative: {moved_count}')
print(f'  Removed: {len(removed_reasons)}')
if removed_reasons:
    print(f'\nRemoval reasons:')
    for r in removed_reasons:
        print(f'  - {r}')

---
# Full run

Run filter on all positions.

In [ ]:
OUTPUT_PATH = os.path.join(DIR, 'data', 'extraction_included_filtered.jsonl')
EXCLUDED_PATH = os.path.join(DIR, 'data', 'extraction_excluded_filtered.jsonl')

# Resume support
already_done = set()
for path in [OUTPUT_PATH, EXCLUDED_PATH]:
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                already_done.add(json.loads(line)['custom_id'])
if already_done:
    print(f'Resuming: {len(already_done)} already done')

n_processed = 0
n_atoms_before = 0
n_atoms_after = 0
n_ctx = 0
n_moved = 0
n_excluded = 0

for row in all_rows:
    if row['custom_id'] in already_done:
        continue

    board = chess.Board(row['fen'])
    san = board.san(chess.Move.from_uci(row['move_uci']))

    result, out_row = run_filter(row)

    old_reasoning = row['extracted'].get('reasoning', [])
    new_reasoning = result['filtered_reasoning']
    n_before = len(old_reasoning)
    n_after = len(new_reasoning)
    n_atoms_before += n_before
    n_atoms_after += n_after
    n_ctx += sum(1 for a in result.get('atoms', []) if a.get('action') == 'contextualize')
    n_moved += len(result.get('alternatives', []))

    out_row['filter_result'] = result.get('atoms', [])

    if not new_reasoning:
        out_row['exclude_reason'] = 'no reasoning atoms after filter'
        with open(EXCLUDED_PATH, 'a') as f:
            f.write(json.dumps(out_row) + '\n')
        n_excluded += 1
        print(f'  [{row["custom_id"]}] {row["game"]} \u2014 {san}: EXCLUDED (0 reasoning atoms)')
    else:
        with open(OUTPUT_PATH, 'a') as f:
            f.write(json.dumps(out_row) + '\n')

    n_processed += 1
    if n_processed % 50 == 0:
        print(f'  {n_processed}/{len(all_rows) - len(already_done)} done '
              f'({n_atoms_before} -> {n_atoms_after} atoms)')
    elif n_before != n_after and new_reasoning:
        print(f'  [{row["custom_id"]}] {row["game"]} \u2014 {san}: {n_before} -> {n_after}')

print(f'\nDone. {n_processed} positions filtered.')
print(f'Included: {n_processed - n_excluded} | Excluded: {n_excluded}')
print(f'Atoms: {n_atoms_before} -> {n_atoms_after} '
      f'({n_atoms_before - n_atoms_after} net removed, '
      f'{(n_atoms_before - n_atoms_after) / max(n_atoms_before, 1) * 100:.1f}%)')
print(f'  Contextualized: {n_ctx}, Moved to alternative: {n_moved}')